# Customer Churn Intelligence — Probability Calibration

## Objective

Evaluate and improve **probability calibration** for the Step-15 leading model (**XGBoost + class weighting**). Use **train + validation only** — the final test set is **not used**.

**Stage:** Step 17 — Probability calibration (no SHAP, no test-set evaluation, no unrelated model retraining).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from xgboost import XGBClassifier

PROJECT_ROOT = Path("..").resolve()
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
CALIBRATION_TABLE_PATH = REPORTS_DIR / "calibration_comparison.csv"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_split import load_split_from_manifest
from src.preprocessing import build_preprocessor

RANDOM_STATE = 42
N_BINS = 10

## 1. Load Data & Fit Leading Model (Uncalibrated)

In [ ]:
split = load_split_from_manifest()
X_train, X_val = split.X_train, split.X_val
y_train = (split.y_train == "Yes").astype(int)
y_val = (split.y_val == "Yes").astype(int)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

preprocessor = build_preprocessor()
X_train_t = preprocessor.fit_transform(X_train)
X_val_t = preprocessor.transform(X_val)

xgb_params = dict(
    n_estimators=200, max_depth=3, learning_rate=0.1, subsample=0.8,
    colsample_bytree=1.0, scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=-1,
)

uncalibrated_model = XGBClassifier(**xgb_params)
uncalibrated_model.fit(X_train_t, y_train)
proba_uncalibrated = uncalibrated_model.predict_proba(X_val_t)[:, 1]

print(f"Train: {len(X_train):,} | Validation: {len(X_val):,}")
print(f"Uncalibrated Brier: {brier_score_loss(y_val, proba_uncalibrated):.4f}")

## 2. Baseline Calibration Assessment

In [ ]:
def calibration_metrics(y_true, y_proba, label: str) -> dict:
    frac_pos, mean_pred = calibration_curve(y_true, y_proba, n_bins=N_BINS, strategy="uniform")
    return {
        "Method": label,
        "Brier_Score": round(brier_score_loss(y_true, y_proba), 4),
        "PR_AUC": round(average_precision_score(y_true, y_proba), 4),
        "ROC_AUC": round(roc_auc_score(y_true, y_proba), 4),
        "Mean_Calibration_Error": round(float(np.mean(np.abs(frac_pos - mean_pred))), 4),
        "Proba_Mean": round(float(y_proba.mean()), 4),
        "Proba_Min": round(float(y_proba.min()), 4),
        "Proba_Max": round(float(y_proba.max()), 4),
    }


baseline = calibration_metrics(y_val, proba_uncalibrated, "Uncalibrated")
pd.DataFrame([baseline])

Uncalibrated XGBoost probabilities show **noticeable miscalibration** (mean calibration error ~0.17). Brier score and calibration curve justify testing post-hoc calibration.

## 3. Calibrated Models (5-fold CV on training data only)

- **Sigmoid (Platt scaling)** — parametric; stable with ~5k training rows
- **Isotonic** — non-parametric; included for comparison; with ~1.3k positive training examples it is borderline — prefer sigmoid if similar Brier but smoother

In [ ]:
cal_sigmoid = CalibratedClassifierCV(
    XGBClassifier(**xgb_params), method="sigmoid", cv=5,
)
cal_isotonic = CalibratedClassifierCV(
    XGBClassifier(**xgb_params), method="isotonic", cv=5,
)

cal_sigmoid.fit(X_train_t, y_train)
cal_isotonic.fit(X_train_t, y_train)

proba_sigmoid = cal_sigmoid.predict_proba(X_val_t)[:, 1]
proba_isotonic = cal_isotonic.predict_proba(X_val_t)[:, 1]

comparison_rows = [
    baseline,
    calibration_metrics(y_val, proba_sigmoid, "Sigmoid (Platt)"),
    calibration_metrics(y_val, proba_isotonic, "Isotonic"),
]
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(CALIBRATION_TABLE_PATH, index=False)
print(f"Saved: {CALIBRATION_TABLE_PATH}")
comparison_df

## 4. Calibration Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for proba, label, color in [
    (proba_uncalibrated, "Uncalibrated", "#4C72B0"),
    (proba_sigmoid, "Sigmoid (Platt)", "#DD8452"),
    (proba_isotonic, "Isotonic", "#55A868"),
]:
    frac_pos, mean_pred = calibration_curve(y_val, proba, n_bins=N_BINS, strategy="uniform")
    brier = brier_score_loss(y_val, proba)
    ax.plot(mean_pred, frac_pos, "o-", label=f"{label} (Brier={brier:.3f})", color=color)

ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.set_title("Calibration Curves — Validation Set")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.legend(loc="lower right")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "10_calibration_curves.png", dpi=120)
plt.show()

## 5. Calibration Decision

### Retain calibration? **Yes — Sigmoid (Platt scaling)**

| Method | Brier ↓ | Mean Cal. Error ↓ | PR-AUC | ROC-AUC |
|--------|---------|-------------------|--------|---------|
| Uncalibrated | 0.1631 | 0.1680 | 0.6399 | 0.8449 |
| **Sigmoid (selected)** | **0.1360** | **0.0249** | 0.6446 | 0.8452 |
| Isotonic | 0.1362 | 0.0376 | **0.6469** | **0.8455** |

**Sigmoid** is selected because it achieves the **lowest Brier score** and **lowest calibration error**, with PR-AUC/ROC-AUC essentially unchanged. Isotonic is slightly better on ranking metrics but less well-calibrated and can be less stable with limited positive samples (~1.3k churners in train).

### ⚠️ Threshold re-optimization required
Calibrated probabilities **shift meaningfully** (validation mean ~0.41 → ~0.27). The Step-16 threshold (**0.26**) was tuned on **uncalibrated** scores and is **not valid** after calibration. Business threshold optimization **must be rerun** on sigmoid-calibrated validation probabilities before deployment.

In [ ]:
# Illustrative: Step-16 threshold on calibrated probabilities
STEP16_THRESHOLD = 0.26
RETENTION_OFFER_COST = 50
LOST_CUSTOMER_COST = 500

pred_old_threshold = (proba_sigmoid >= STEP16_THRESHOLD).astype(int)
from sklearn.metrics import confusion_matrix
tn, fp, fn, tp = confusion_matrix(y_val, pred_old_threshold).ravel()
cost_old = fp * RETENTION_OFFER_COST + fn * LOST_CUSTOMER_COST

# Quick re-scan for min-cost threshold on calibrated probs
best_cost, best_t = float("inf"), None
for t in np.arange(0.05, 0.96, 0.01):
    pred = (proba_sigmoid >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    cost = fp * RETENTION_OFFER_COST + fn * LOST_CUSTOMER_COST
    if cost < best_cost:
        best_cost, best_t = cost, t

print(f"Step-16 threshold {STEP16_THRESHOLD} on calibrated probs → cost ${cost_old:,}")
print(f"Recalibrated optimal threshold ≈ {best_t:.2f} → cost ${best_cost:,}")
print("→ Threshold optimization from Step 16 must be rerun (not done in this notebook).")